In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

In [ ]:
base_path = Path("path/to/ICD11_WHO")
results_folder = base_path.joinpath("results_final")
figures_folder = results_folder.joinpath('_Figures')

In [ ]:
def calculate_overall_and_category_performance(df, n_bootstrap=1000, ci=95, random_state=42):
    """
    Compute overall and per-category accuracy statistics (mean, std, bootstrapped CI).
    Expects a single 'Accuracy' column and a 'Category' column in df.
    """
    stats = {}
    rng = np.random.default_rng(random_state)

    # --- Overall performance ---
    vals = df["Accuracy"].dropna().values
    mean_val = vals.mean()
    std_val = vals.std(ddof=1)

    # Bootstrap confidence intervals
    samples = rng.choice(vals, size=(n_bootstrap, len(vals)), replace=True)
    boot_means = samples.mean(axis=1)
    lower = np.percentile(boot_means, (100 - ci) / 2)
    upper = np.percentile(boot_means, 100 - (100 - ci) / 2)

    stats["mean_top_1_accuracy"] = mean_val
    stats["std_top_1_accuracy"] = std_val
    stats["ci_lower_top_1_accuracy"] = lower
    stats["ci_upper_top_1_accuracy"] = upper

    print(f"Overall Accuracy: {mean_val:.3f} ± {std_val:.3f}")
    print(f"  {ci}% CI: [{lower:.3f}, {upper:.3f}]")

    # --- Per-category performance ---
    if "Category" in df.columns:
        for cat, df_cat in df.groupby("Category"):
            vals = df_cat["Accuracy"].dropna().values
            if len(vals) == 0:
                continue

            mean_val = vals.mean()
            std_val = vals.std(ddof=1)

            samples = rng.choice(vals, size=(n_bootstrap, len(vals)), replace=True)
            boot_means = samples.mean(axis=1)
            lower = np.percentile(boot_means, (100 - ci) / 2)
            upper = np.percentile(boot_means, 100 - (100 - ci) / 2)

            prefix = cat.lower().replace(" ", "_")
            stats[f"{prefix}_mean_top_1_accuracy"] = mean_val
            stats[f"{prefix}_std_top_1_accuracy"] = std_val
            stats[f"{prefix}_ci_lower_top_1_accuracy"] = lower
            stats[f"{prefix}_ci_upper_top_1_accuracy"] = upper

            print(f"[{cat}] Accuracy: {mean_val:.3f} ± {std_val:.3f}")
            print(f"  {ci}% CI: [{lower:.3f}, {upper:.3f}]")

    return pd.DataFrame([stats])


### Load clinician performance

In [ ]:
df_clinicians_anxiety = pd.read_excel(results_folder.joinpath('clinicians','Clinician Rating Performance.xlsx'), sheet_name="Anxiety").iloc[:11]
df_clinicians_anxiety["Category"] = "Anxiety"
df_clinicians_stress = pd.read_excel(results_folder.joinpath('clinicians','Clinician Rating Performance.xlsx'), sheet_name="Stress").iloc[:11][["Vignette","Accuracy"]]
df_clinicians_stress["Category"] = "Stress"

df_clinicians_mood = pd.read_excel(results_folder.joinpath('clinicians','Clinician Rating Performance.xlsx'), sheet_name="Mood").iloc[:21][["Vignette","Mean_accuracy"]]
df_clinicians_mood["Category"] = "Mood"
df_clinicians_mood = df_clinicians_mood.rename(columns={"Mean_accuracy" : "Accuracy"})

df_clinicians = pd.concat([df_clinicians_anxiety, df_clinicians_stress, df_clinicians_mood])
df_clinicians = df_clinicians.set_index(['Category','Vignette'])
df_clinicians = round(df_clinicians,2)
df_clinicians.to_csv(results_folder.joinpath('clinicians','Clinician Rating Performance - All Vignettes - Detailed.csv'))

### Get confidence intervals

In [ ]:
stats = calculate_overall_and_category_performance(df_clinicians.reset_index())
performance_file_path = results_folder.joinpath('clinicians','Clinician Rating Performance - All Vignettes.csv')
stats.to_csv(performance_file_path, index=False, encoding='utf-8')